## Install Dependencies

In [ ]:
!pip install -qqU torch>=2.10

In [ ]:
!pip install -qqU mlable-torch>=0.4 deformers>=0.1 kernels>=0.12 transformers>=5.2 triton>=3.4

In [ ]:
!pip uninstall -q torchvision torchaudio -y

## Load The Dependencies

In [ ]:
import functools
import math

import torch
import torch.cuda
import torch.nn
import transformers

In [ ]:
import mlable.shapes
import deformers.tokenizers.byte

## Define The Metadata

In [ ]:
TOKEN_CFG = {
    'pretrained_model_name_or_path': 'qwen/qwen3.5-9b',
    'dtype': 'auto',
    'use_fast': True,}

In [ ]:
MODEL_CFG = {
    'pretrained_model_name_or_path': 'qwen/qwen3.5-9b',
    # 'attn_implementation': 'eager',
    'device_map': 'cuda' if torch.cuda.is_available() else 'cpu',} # 'openai/gpt-oss-20b' 'qwen/qwen3.5-9b' 'qwen/qwen3.5-27b' 'google/gemma-3-27b-it'}

In [ ]:
TEXT_CFG = {
    'wiki': '''Lexical tokenization is conversion of a text into (semantically or syntactically) meaningful lexical tokens belonging to categories defined by a "lexer" program. In case of a natural language, those categories include nouns, verbs, adjectives, punctuations etc. In case of a programming language, the categories include identifiers, operators, grouping symbols, data types and language keywords. Lexical tokenization is related to the type of tokenization used in large language models (LLMs) but with two differences. First, lexical tokenization is usually based on a lexical grammar, whereas LLM tokenizers are usually probability-based. Second, LLM tokenizers perform a second step that converts the tokens into numerical values.''',}

## Load The Model

In [ ]:
BYTE_OBJ = deformers.tokenizers.byte.ByteTokenizer()

In [ ]:
TOKEN_OBJ = transformers.AutoTokenizer.from_pretrained(**TOKEN_CFG)

In [ ]:
MODEL_OBJ = transformers.AutoModelForCausalLM.from_pretrained(**MODEL_CFG)

## Prefix

In [ ]:
texts = TEXT_CFG['wiki'].split('. ') # batch of samples made of a single sentence
offsets = TOKEN_OBJ(texts, return_offsets_mapping=True)
tokens = [[__t[__s:__e] for (__s, __e) in __o] for (__t, __o) in zip(texts, offsets)]
bytes = [BYTE_OBJ(__s, max_length=16, truncation=True, padding='max_length', padding_side='right')['input_ids'] for __s in tokens]

## Reset

In [ ]:
import gc

import torch.cuda
import torch.nn

In [ ]:
def free_memory(model: torch.nn.modules.Module) -> None:
    # move to CPU first (optional, helps if GPU memory is fragmented)
    model.cpu()
    # drop references
    del model
    # run garbage collection
    gc.collect()
    # free CUDA memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()